In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import torchvision.utils as vutils

# --- Dataset Class ---
class PolymerDataset(Dataset):
    def __init__(self, input_dir, target_dir, size=(512, 512)):
        self.input_dir = input_dir
        self.target_dir = target_dir
        self.size = size
        self.filenames = sorted(os.listdir(input_dir))
        
        self.transform = transforms.Compose([
            transforms.Resize(size),
            transforms.ToTensor(),
        ])
    
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        input_path = os.path.join(self.input_dir, self.filenames[idx])
        target_path = os.path.join(self.target_dir, self.filenames[idx])
        
        input_img = Image.open(input_path).convert('L')
        target_img = Image.open(target_path).convert('L')
        
        input_tensor = self.transform(input_img)
        target_tensor = self.transform(target_img)
        
        input_tensor = (input_tensor > 0.5).float() * 2 - 1
        target_tensor = (target_tensor > 0.5).float() * 2 - 1
        
        return input_tensor, target_tensor

In [2]:
# --- UNet Generator ---
class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()
        self.conv_in = nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1, bias=False)
        
        # Downsampling
        self.down1 = nn.Sequential(
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128)
        )
        self.down2 = self._make_down_block(128, 256)
        self.down3 = self._make_down_block(256, 512)
        self.down4 = self._make_down_block(512, 512)
        self.down5 = self._make_down_block(512, 512)
        self.down6 = self._make_down_block(512, 512)

        # Middle
        self.middle = nn.Sequential(
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 512, kernel_size=4, stride=2, padding=1, bias=False),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(512, 512, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512)
        )

        # Upsampling
        self.up1 = self._make_up_block(1024, 512, dropout=True)
        self.up2 = self._make_up_block(1024, 512, dropout=True)
        self.up3 = self._make_up_block(1024, 512, dropout=True)
        self.up4 = self._make_up_block(1024, 256)
        self.up5 = self._make_up_block(512, 128)
        self.up6 = self._make_up_block(256, 64)

        self.outermost = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def _make_down_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )

    def _make_up_block(self, in_channels, out_channels, dropout=False):
        layers = [
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        ]
        if dropout:
            layers.append(nn.Dropout(0.5))
        return nn.Sequential(*layers)

    def forward(self, x):
        # Encoder
        x0 = self.conv_in(x)
        x1 = self.down1(x0)
        x2 = self.down2(x1)
        x3 = self.down3(x2)
        x4 = self.down4(x3)
        x5 = self.down5(x4)
        x6 = self.down6(x5)

        # Middle
        x = self.middle(x6)

        # Decoder with skip connections
        x = self.up1(torch.cat([x, x6], dim=1))
        x = self.up2(torch.cat([x, x5], dim=1))
        x = self.up3(torch.cat([x, x4], dim=1))
        x = self.up4(torch.cat([x, x3], dim=1))
        x = self.up5(torch.cat([x, x2], dim=1))
        x = self.up6(torch.cat([x, x1], dim=1))
        
        return self.outermost(torch.cat([x, x0], dim=1))

In [3]:
# --- PatchGAN Discriminator ---
class PatchGAN(nn.Module):
    def __init__(self):
        super(PatchGAN, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(256, 512, kernel_size=4, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1)
        )

    def forward(self, x):
        return self.net(x)

# --- Learning Rate Scheduler ---
class LambdaLR:
    def __init__(self, n_epochs, offset, decay_start_epoch):
        self.n_epochs = n_epochs
        self.offset = offset
        self.decay_start_epoch = decay_start_epoch

    def step(self, epoch):
        return 1.0 - max(0, epoch + self.offset - self.decay_start_epoch) / (self.n_epochs - self.decay_start_epoch)

# --- Visualization Function ---
def visualize_results(inputs, targets, outputs, epoch, save_dir="training_results"):
    os.makedirs(save_dir, exist_ok=True)
    
    # Denormalize images
    inputs = (inputs.cpu().numpy() + 1) / 2
    targets = (targets.cpu().numpy() + 1) / 2
    outputs = (outputs.cpu().numpy() + 1) / 2
    
    plt.figure(figsize=(15, 5))
    for i in range(min(3, inputs.shape[0])):  # Show max 3 examples
        plt.subplot(3, 3, i*3+1)
        plt.imshow(inputs[i][0], cmap='gray')
        plt.title("Input")
        plt.axis('off')
        
        plt.subplot(3, 3, i*3+2)
        plt.imshow(targets[i][0], cmap='gray')
        plt.title("Target")
        plt.axis('off')
        
        plt.subplot(3, 3, i*3+3)
        plt.imshow(outputs[i][0], cmap='gray')
        plt.title("Output")
        plt.axis('off')
    
    plt.suptitle(f"Epoch {epoch+1}")
    plt.savefig(f"{save_dir}/epoch_{epoch+1}.png")
    plt.close()

In [4]:
# Load dataset
#input_dir = '/kaggle/input/dataset1/sliced images/input'
#target_dir = '/kaggle/input/dataset1/sliced images/target'
    

# Split dataset
#dataset = PolymerDataset(input_dir, target_dir)

#train_size = int(0.8 * len(dataset))
#test_size = len(dataset) - train_size
#gen = torch.Generator()
#gen.manual_seed(30)
#train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=gen)
#train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
#test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [152]:
import os
import re
import random
from torch.utils.data import DataLoader
from collections import defaultdict

# Пути к данным
input_dir = '/kaggle/input/dataset1/sliced images/input'
target_dir = '/kaggle/input/dataset1/sliced images/target'

# Получаем список файлов
all_filenames = [f for f in os.listdir(input_dir) if f.lower().endswith('.png')]

# Регулярка для получения базового ID до угла поворота: например, 1_1, 2_5 и т.д.
pattern = re.compile(r'^(.+?)(?:_[\+\-]?\d+)?\.png$')

# Группируем по "базовому" имени (например, '1_1' для '1_1.png', '1_1_180.png', и т.д.)
groups = defaultdict(list)
for fname in all_filenames:
    base = fname.split('.')[0]
    base_id = base.split('_')[0] + "_" + base.split('_')[1]  # '1_1' из '1_1_180'
    groups[base_id].append(fname)

# Получаем список базовых ID и делим их
all_ids = list(groups.keys())
random.seed(30)
random.shuffle(all_ids)

train_ids = set(all_ids[:int(0.8 * len(all_ids))])
test_ids = set(all_ids[int(0.8 * len(all_ids)):])

# Списки файлов по группам
train_files = [fname for gid in train_ids for fname in groups[gid]]
test_files  = [fname for gid in test_ids  for fname in groups[gid]]

# Создаём датасеты
train_dataset = PolymerDataset(input_dir, target_dir)
test_dataset  = PolymerDataset(input_dir, target_dir)

# Переопределим __getitem__ в dataset, чтобы использовать только нужные файлы
train_dataset.filenames = sorted(train_files)
test_dataset.filenames  = sorted(test_files)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False)


In [193]:
csv_path = 'test_files.csv'

with open(csv_path, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['filename'])
    for file in test_dataset.filenames:
        writer.writerow([file])

print(f"Файл сохранён: {csv_path}")

Файл сохранён: test_files.csv


In [6]:
import re

# Регулярка для базового имени: два числа через подчёркивание в начале строки
base_pattern = re.compile(r'^(\d+_\d+)')

# Словарь: base_name -> set(версия)
groups = {}

for fname in train_files:
    match = base_pattern.match(fname)
    if match:
        base_name = match.group(1)  # например '1_1'
        if base_name not in groups:
            groups[base_name] = set()
        
        if fname == f"{base_name}.png":
            groups[base_name].add('base')
        elif fname == f"{base_name}_180.png":
            groups[base_name].add('180')
        elif fname == f"{base_name}_+90.png":
            groups[base_name].add('+90')
        elif fname == f"{base_name}_-90.png":
            groups[base_name].add('-90')
        else:
            groups[base_name].add('other')

required_versions = {'base', '180', '+90', '-90'}

missing_versions = {}
for base, versions in groups.items():
    missing = required_versions - versions
    if missing:
        missing_versions[base] = missing

if missing_versions:
    print("Базовые имена с отсутствующими версиями:")
    for base, miss in missing_versions.items():
        print(f"{base}: отсутствуют версии {', '.join(sorted(miss))}")
else:
    print("Все базовые имена имеют исходник и все три версии.")


Все базовые имена имеют исходник и все три версии.


In [7]:
import re
import csv

def sort_key(fname):
    pattern = re.compile(r'^(.*?)(?:_([\+\-]?\d+))?\.png$')
    match = pattern.match(fname)
    if not match:
        return (fname, 0)  # если не совпало — по имени и 0

    base = match.group(1)
    angle_str = match.group(2)
    angle = 0  # версия без угла - самый первый
    if angle_str is not None:
        angle = int(angle_str)
    return (base, angle)

# Сортируем с помощью кастомного ключа
train_files_sorted = sorted(train_files, key=sort_key)

# Запись в CSV, как было
csv_path = 'train_files.csv'
with open(csv_path, mode='w', newline='') as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(['filename'])
    for fname in train_files_sorted:
        writer.writerow([fname])

print(f'Train filenames saved to {csv_path}')



Train filenames saved to train_files.csv


In [153]:
len(test_loader)

66

In [154]:
len(train_loader)

264

In [10]:
# --- Training Function with Visualization ---
def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Initialize models
    netG = UNet().to(device)
    netD = PatchGAN().to(device)
    
    # Loss functions
    criterion_GAN = nn.BCEWithLogitsLoss()
    criterion_L1 = nn.L1Loss()
    lambda_L1 = 20
    
    # Optimizers
    optimizer_G = torch.optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))
    optimizer_D = torch.optim.Adam(netD.parameters(), lr=0.0001, betas=(0.5, 0.999))
    
    # Learning rate schedulers
    lr_scheduler_G = torch.optim.lr_scheduler.LambdaLR(
        optimizer_G, lr_lambda=LambdaLR(100, 0, 50).step)
    lr_scheduler_D = torch.optim.lr_scheduler.LambdaLR(
        optimizer_D, lr_lambda=LambdaLR(100, 0, 50).step)

    
    # Create fixed batch for visualization
    fixed_inputs, fixed_targets = next(iter(test_loader))
    fixed_inputs = fixed_inputs.to(device)
    
    # History tracking
    history = {
        'epoch': [],
        'G_loss': [],
        'D_loss': [],
        'D_real': [],
        'D_fake': [],
        'D_x': [],       # D(x) - average output on real images
        'D_G_z': [],     # D(G(z)) - average output on fake images
        'val_loss': []
    }
    
    # Training loop
    for epoch in range(100):
        netG.train()
        netD.train()
        
        # Initialize accumulators
        running_loss_G = 0.0
        running_loss_D = 0.0
        running_loss_D_real = 0.0
        running_loss_D_fake = 0.0
        running_D_x = 0.0
        running_D_G_z = 0.0
        d_steps = 0
        
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/100')
        for i, (input_imgs, target_imgs) in enumerate(progress_bar):
            input_imgs, target_imgs = input_imgs.to(device), target_imgs.to(device)
            
            # Train Generator
            optimizer_G.zero_grad()
            fake_B = netG(input_imgs)
            fake_AB = torch.cat((input_imgs, fake_B), 1)
            
            pred_fake = netD(fake_AB)
            loss_G_GAN = criterion_GAN(pred_fake, torch.ones_like(pred_fake))
            loss_G_L1 = criterion_L1(fake_B, target_imgs) * lambda_L1
            loss_G = loss_G_GAN + loss_G_L1
            loss_G.backward()
            optimizer_G.step()
            
            running_loss_G += loss_G.item()
            running_D_G_z += torch.sigmoid(pred_fake).mean().item()
            
            # Train Discriminator (every 2nd step)
            if i % 2 == 0:
                optimizer_D.zero_grad()
                
                # Real images
                real_AB = torch.cat((input_imgs, target_imgs), 1)
                pred_real = netD(real_AB)
                real_labels = torch.rand_like(pred_real)*0.1 + 0.9
                loss_D_real = criterion_GAN(pred_real, real_labels)
                
                # Fake images
                fake_AB = torch.cat((input_imgs, fake_B.detach()), 1)
                pred_fake = netD(fake_AB)
                fake_labels = torch.rand_like(pred_fake)*0.1
                loss_D_fake = criterion_GAN(pred_fake, fake_labels)
                
                loss_D = (loss_D_real + loss_D_fake) * 0.5
                loss_D.backward()
                optimizer_D.step()
                
                running_loss_D += loss_D.item()
                running_loss_D_real += loss_D_real.item()
                running_loss_D_fake += loss_D_fake.item()
                running_D_x += torch.sigmoid(pred_real).mean().item()
                d_steps += 1
            
            # Update progress bar
            progress_bar.set_postfix({
                'G': f'{running_loss_G/(i+1):.4f}',
                'D': f'{running_loss_D/d_steps:.4f}' if d_steps > 0 else 'skipped',
                'D(x)': f'{running_D_x/d_steps:.3f}' if d_steps > 0 else '-',
                'D(G(z))': f'{running_D_G_z/(i+1):.3f}'
            })
            
        # Update learning rates
        lr_scheduler_G.step()
        lr_scheduler_D.step()
        
        # Validation
        netG.eval()
        val_loss = 0
        with torch.no_grad():
            for input_imgs, target_imgs in test_loader:
                input_imgs, target_imgs = input_imgs.to(device), target_imgs.to(device)
                fake_B = netG(input_imgs)
                val_loss += criterion_L1(fake_B, target_imgs).item()
            
            # Generate visualization
            fixed_outputs = netG(fixed_inputs)
            visualize_results(fixed_inputs[:3], fixed_targets[:3].to(device), fixed_outputs[:3], epoch)
        
        # Calculate averages
        avg_loss_G = running_loss_G / len(train_loader)
        avg_loss_D = running_loss_D / d_steps if d_steps > 0 else 0
        avg_loss_D_real = running_loss_D_real / d_steps if d_steps > 0 else 0
        avg_loss_D_fake = running_loss_D_fake / d_steps if d_steps > 0 else 0
        avg_D_x = running_D_x / d_steps if d_steps > 0 else 0
        avg_D_G_z = running_D_G_z / len(train_loader)
        avg_val_loss = val_loss / len(test_loader)
        
        # Store history
        history['epoch'].append(epoch+1)
        history['G_loss'].append(avg_loss_G)
        history['D_loss'].append(avg_loss_D)
        history['D_real'].append(avg_loss_D_real)
        history['D_fake'].append(avg_loss_D_fake)
        history['D_x'].append(avg_D_x)
        history['D_G_z'].append(avg_D_G_z)
        history['val_loss'].append(avg_val_loss)
        
        # Print epoch summary
        print(f'\nEpoch {epoch+1} Summary:')
        print(f'G_loss: {avg_loss_G:.4f} | D_loss: {avg_loss_D:.4f}')
        print(f'D_real: {avg_loss_D_real:.4f} | D_fake: {avg_loss_D_fake:.4f}')
        print(f'D(x): {avg_D_x:.3f} | D(G(z)): {avg_D_G_z:.3f}')
        print(f'Val Loss: {avg_val_loss:.4f}')
        
        # Save models
        if (epoch + 1) % 10 == 0:
            torch.save(netG.state_dict(), f'generator_epoch_{epoch+1}.pth')
            torch.save(netD.state_dict(), f'discriminator_epoch_{epoch+1}.pth')
    
    # Save final models
    torch.save(netG.state_dict(), 'generator_final.pth')
    torch.save(netD.state_dict(), 'discriminator_final.pth')
    
    # Plot training history
    plot_training_history(history)

def plot_training_history(history):
    plt.figure(figsize=(15, 10))
    
    # Loss plot
    plt.subplot(2, 2, 1)
    plt.plot(history['epoch'], history['G_loss'], label='Generator Loss')
    plt.plot(history['epoch'], history['D_loss'], label='Discriminator Loss')
    plt.plot(history['epoch'], history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Losses')
    plt.legend()
    
    # D(x) and D(G(z)) plot
    plt.subplot(2, 2, 2)
    plt.plot(history['epoch'], history['D_x'], label='D(x) - Real')
    plt.plot(history['epoch'], history['D_G_z'], label='D(G(z)) - Fake')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.title('Discriminator Output Scores')
    plt.legend()
    
    # D components plot
    plt.subplot(2, 2, 3)
    plt.plot(history['epoch'], history['D_real'], label='D Real Loss')
    plt.plot(history['epoch'], history['D_fake'], label='D Fake Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Discriminator Component Losses')
    plt.legend()
    
    # Save plots
    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.close()

if __name__ == "__main__":
    train()

Epoch 1/100: 100%|██████████| 264/264 [01:36<00:00,  2.73it/s, G=7.9461, D=0.5677, D(x)=0.597, D(G(z))=0.392]



Epoch 1 Summary:
G_loss: 7.9461 | D_loss: 0.5677
D_real: 0.5772 | D_fake: 0.5582
D(x): 0.597 | D(G(z)): 0.392
Val Loss: 1.0100


Epoch 2/100: 100%|██████████| 264/264 [01:19<00:00,  3.33it/s, G=8.0056, D=0.5158, D(x)=0.647, D(G(z))=0.354]



Epoch 2 Summary:
G_loss: 8.0056 | D_loss: 0.5158
D_real: 0.5373 | D_fake: 0.4944
D(x): 0.647 | D(G(z)): 0.354
Val Loss: 0.3250


Epoch 3/100: 100%|██████████| 264/264 [01:19<00:00,  3.32it/s, G=8.0331, D=0.5704, D(x)=0.620, D(G(z))=0.372]


...


Epoch 98 Summary:
G_loss: 6.2234 | D_loss: 0.4558
D_real: 0.4565 | D_fake: 0.4552
D(x): 0.671 | D(G(z)): 0.329
Val Loss: 0.3338


Epoch 99/100: 100%|██████████| 264/264 [01:18<00:00,  3.34it/s, G=6.2052, D=0.4499, D(x)=0.675, D(G(z))=0.326]



Epoch 99 Summary:
G_loss: 6.2052 | D_loss: 0.4499
D_real: 0.4499 | D_fake: 0.4498
D(x): 0.675 | D(G(z)): 0.326
Val Loss: 0.3374


Epoch 100/100: 100%|██████████| 264/264 [01:19<00:00,  3.33it/s, G=6.1922, D=0.4470, D(x)=0.676, D(G(z))=0.327]



Epoch 100 Summary:
G_loss: 6.1922 | D_loss: 0.4470
D_real: 0.4472 | D_fake: 0.4468
D(x): 0.676 | D(G(z)): 0.327
Val Loss: 0.3350


In [194]:
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [203]:
print(test_loader.dataset.filenames)

['1_10.png', '1_10_+90.png', '1_10_-90.png', '1_10_180.png', '1_11.png', '1_11_+90.png', '1_11_-90.png', '1_11_180.png', '1_15.png', '1_15_+90.png', '1_15_-90.png', '1_15_180.png', '1_16.png', '1_16_+90.png', '1_16_-90.png', '1_16_180.png', '1_18.png', '1_18_+90.png', '1_18_-90.png', '1_18_180.png', '1_20.png', '1_20_+90.png', '1_20_-90.png', '1_20_180.png', '1_21.png', '1_21_+90.png', '1_21_-90.png', '1_21_180.png', '1_27.png', '1_27_+90.png', '1_27_-90.png', '1_27_180.png', '1_37.png', '1_37_+90.png', '1_37_-90.png', '1_37_180.png', '1_46.png', '1_46_+90.png', '1_46_-90.png', '1_46_180.png', '1_5.png', '1_5_+90.png', '1_5_-90.png', '1_5_180.png', '1_6.png', '1_6_+90.png', '1_6_-90.png', '1_6_180.png', '2_18.png', '2_18_+90.png', '2_18_-90.png', '2_18_180.png', '2_20.png', '2_20_+90.png', '2_20_-90.png', '2_20_180.png', '2_25.png', '2_25_+90.png', '2_25_-90.png', '2_25_180.png', '2_28.png', '2_28_+90.png', '2_28_-90.png', '2_28_180.png', '2_42.png', '2_42_+90.png', '2_42_-90.png', '2_

In [204]:
len(test_loader)

264

In [215]:
import csv
import os


test_dataset1 = test_loader.dataset  # Получаем датасет из loader

csv_path = 'test_files_loader.csv'

with open(csv_path, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['filename'])
    for file_path in test_dataset1.filenames:
        writer.writerow([os.path.basename(file_path)])

print(f"CSV файл {csv_path}")


CSV файл test_files_loader.csv


In [211]:
import torch
from torch.utils.data import DataLoader, random_split
import torchvision.utils as vutils
import os
from tqdm import tqdm
from matplotlib import pyplot as plt
from PIL import Image
import numpy as np

def inference(checkpoint_path, save_dir='predictions', num_samples=len(test_loader)):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    netG = UNet().to(device)
    netG.load_state_dict(torch.load(checkpoint_path, map_location=device))
    netG.eval()

    os.makedirs(save_dir, exist_ok=True)
    mask_dir = os.path.join(save_dir, 'predicted_masks')
    os.makedirs(mask_dir, exist_ok=True)

    # Проверим, что filenames есть в датасете, если нет - ошибка
    if not hasattr(test_dataset, 'filenames'):
        raise AttributeError("test_dataset не содержит атрибут filenames")

    with torch.no_grad():
        for idx, (input_img, target_img) in tqdm(enumerate(test_loader), total=len(test_loader)):
            input_img = input_img.to(device)
            fake_mask = netG(input_img)

            base_name = os.path.splitext(os.path.basename(test_dataset.filenames[idx]))[0]

            fake_mask_img = fake_mask.squeeze(0).squeeze(0).cpu()
            target_vis = target_img.squeeze(0).squeeze(0).cpu()
            input_vis = input_img.squeeze(0).cpu()

            plt.imsave(os.path.join(mask_dir, f'{base_name}_pred.png'), fake_mask_img, cmap='gray')

            alpha = 0.5
            input_np = (input_vis.squeeze().cpu().numpy() + 1) / 2
            input_np = np.clip(input_np, 0, 1)
            mask_np = np.clip(fake_mask_img.cpu().numpy(), 0, 1)
            overlay = np.clip((1 - alpha) * input_np + alpha * mask_np, 0, 1)

            plt.imsave(os.path.join(mask_dir, f'{base_name}_overlay.png'), overlay, cmap='gray')

            fig, axs = plt.subplots(1, 3, figsize=(12, 4))
            axs[0].imshow(input_np, cmap='gray')
            axs[0].set_title('Input')
            axs[1].imshow(target_vis, cmap='gray')
            axs[1].set_title('Ground Truth')
            axs[2].imshow(fake_mask_img, cmap='gray')
            axs[2].set_title('Predicted Mask')
            for ax in axs:
                ax.axis('off')
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f'{base_name}_triplet.png'))
            plt.close()

            if idx + 1 >= num_samples:
                break

if __name__ == '__main__':
    inference(
        checkpoint_path='generator_final.pth',
        save_dir='predictions1',
        num_samples=len(test_loader)
    )

/tmp/ipykernel_31/3016503088.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  netG.load_state_dict(torch.load(checkpoint_path, map_location=device))
100%|█████████▉| 263

In [212]:
len(test_loader)

264

dilation_ratio=0.01 — значит, контур расширяется примерно на 1% от среднего размера изображения. Изображение 512x512, то 1% от 512 ≈ 5 пикселей.

Такой расширенный контур даёт "зону допуска", чтобы мелкие сдвиги границ не сильно влияли на метрику.

In [213]:
import torch
from torch.utils.data import DataLoader, random_split
import os
from tqdm import tqdm
from matplotlib import pyplot as plt
from PIL import Image
import numpy as np
from skimage.metrics import structural_similarity as ssim_func
from math import log10
import csv

from skimage import morphology
from scipy.ndimage import binary_dilation
from scipy.spatial.distance import cdist

def boundary_mask(mask, dilation_ratio=0.01):
    """
    Возвращает маску контура (границы) бинарного изображения.
    dilation_ratio — относительная толщина границы (от размера изображения)
    """
    h, w = mask.shape
    dilation = max(1, int(round(dilation_ratio * (h + w) / 2)))

    # Эрозия маски, чтобы выделить контур
    eroded = morphology.binary_erosion(mask)
    boundary = mask ^ eroded  # XOR - граница
    
    # Для устойчивости расширение границу (чтобы разрешить небольшие смещения)
    boundary_dilated = binary_dilation(boundary, structure=np.ones((dilation, dilation)))

    return boundary, boundary_dilated

def boundary_f1_score(pred_mask, true_mask, dilation_ratio=0.01):
    """
    Вычисляет Boundary F1-score между предсказанной и эталонной масками.
    dilation_ratio — максимальное расстояние для совпадения точек границ.
    """

    pred_mask = pred_mask.astype(bool)
    true_mask = true_mask.astype(bool)

    pred_boundary, pred_dilated = boundary_mask(pred_mask, dilation_ratio)
    true_boundary, true_dilated = boundary_mask(true_mask, dilation_ratio)

    # True positives: точки границы предсказания, которые близки к границе эталона
    tp = np.sum(pred_boundary & true_dilated)
    # False positives: точки предсказания, которые не близки к эталону
    fp = np.sum(pred_boundary & (~true_dilated))

    # True positives: точки границы эталона, которые близки к предсказанию
    tp2 = np.sum(true_boundary & pred_dilated)
    # False negatives: точки эталона, которые не близки к предсказанию
    fn = np.sum(true_boundary & (~pred_dilated))

    # Precision и recall для границ
    precision = tp / (tp + fp + 1e-8)
    recall = tp2 / (tp2 + fn + 1e-8)

    if precision + recall == 0:
        return 0.0

    f1 = 2 * precision * recall / (precision + recall)
    return f1

def compute_metrics(pred, target):
    pred = (pred > 0).astype(np.uint8)
    target = (target > 0).astype(np.uint8)

    intersection = np.logical_and(pred, target).sum()
    union = np.logical_or(pred, target).sum()

    dice = 2 * intersection / (pred.sum() + target.sum() + 1e-8)
    iou = intersection / (union + 1e-8)
    mae = np.mean(np.abs(pred - target))
    rmse = np.sqrt(np.mean((pred - target) ** 2))
    ssim = ssim_func(target, pred, data_range=1)

    mse = np.mean((pred - target) ** 2)
    psnr = 20 * log10(1.0) - 10 * log10(mse + 1e-8)  # Assuming pixel values in [0, 1]

    boundary_f1 = boundary_f1_score(pred, target)
    
    return dice, iou, mae, rmse, ssim, psnr, boundary_f1

def inference_with_metrics(checkpoint_path, save_dir='predictions'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Загрузка модели
    netG = UNet().to(device)
    netG.load_state_dict(torch.load(checkpoint_path, map_location=device))
    netG.eval()

    os.makedirs(save_dir, exist_ok=True)

    total_dice = total_iou = total_mae = total_rmse = total_ssim = total_psnr = total_boundary_f1 = 0

    with torch.no_grad():
        for idx, (input_img, target_img) in tqdm(enumerate(test_loader), total=len(test_loader)):
            input_img = input_img.to(device)
            fake_mask = netG(input_img)

            fake_mask_np = fake_mask.squeeze().cpu().numpy()
            target_np = target_img.squeeze().cpu().numpy()

            # Бинаризация
            fake_mask_np_bin = (fake_mask_np > 0).astype(np.uint8)
            target_np_bin = (target_np > 0).astype(np.uint8)

            # Метрики
            dice, iou, mae, rmse, ssim, psnr, boundary_f1 = compute_metrics(fake_mask_np_bin, target_np_bin)
            total_dice += dice
            total_iou += iou
            total_mae += mae
            total_rmse += rmse
            total_ssim += ssim
            total_psnr += psnr
            total_boundary_f1 += boundary_f1

    N = len(test_loader)
    avg_dice = total_dice / N
    avg_iou = total_iou / N
    avg_mae = total_mae / N
    avg_rmse = total_rmse / N
    avg_ssim = total_ssim / N
    avg_psnr = total_psnr / N
    avg_boundary_f1 = total_boundary_f1 / N

    print(f"\n Metrics on Test Set:")
    print(f"Dice:  {avg_dice:.4f}")
    print(f"IoU:   {avg_iou:.4f}")
    print(f"MAE:   {avg_mae:.4f}")
    print(f"RMSE:  {avg_rmse:.4f}")
    print(f"SSIM:  {avg_ssim:.4f}")
    print(f"PSNR:  {avg_psnr:.4f}")
    print(f"Boundary F1:    {avg_boundary_f1:.4f}")

    csv_path = os.path.join(save_dir, 'metrics.csv')
    with open(csv_path, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Metric', 'Value'])
        writer.writerow(['Dice', avg_dice])
        writer.writerow(['IoU', avg_iou])
        writer.writerow(['MAE', avg_mae])
        writer.writerow(['RMSE', avg_rmse])
        writer.writerow(['SSIM', avg_ssim])
        writer.writerow(['PSNR', avg_psnr])
        writer.writerow(['Boundary F1', avg_boundary_f1])

if __name__ == '__main__':
    inference_with_metrics(
        checkpoint_path='generator_final.pth',
        save_dir='predictions1'
    )

/tmp/ipykernel_31/621427038.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  netG.load_state_dict(torch.load(checkpoint_path, map_location=device))
100%|██████████| 264/


 Metrics on Test Set:
Dice:  0.8305
IoU:   0.7235
MAE:   21.3447
RMSE:  0.3943
SSIM:  0.5104
PSNR:  8.3266
Boundary F1:    0.8215


In [214]:
import shutil

shutil.make_archive('/kaggle/working/predictions1', 'zip', '/kaggle/working/predictions1')

'/kaggle/working/predictions1.zip'

In [142]:
shutil.make_archive('/kaggle/working/training_results', 'zip', '/kaggle/working/training_results')

'/kaggle/working/training_results.zip'

In [168]:
import os

file_path = '/kaggle/working/training_results.zip'  # Замените на свой путь

if os.path.exists(file_path):
    os.remove(file_path)
    print(f"Удалено: {file_path}")
else:
    print(f" Файл не найден: {file_path}")

Удалено: /kaggle/working/training_results.zip
